# QueryCampus RAG Prototyping & Experimentation
This notebook evaluates different chunking sizes, embedding models, and retrieval strategies.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from src.ingest import DocumentIngestor, RecursiveTextSplitter
from src.vector_store import ChromaVectorStore
from src.retrieval import CampusRetriever
from src.generator import CampusGenerator

## 1. Chunk Size & Overlap Sensitivity Experiments

In [ ]:
test_text = (
    "Every registered student is required to maintain a minimum attendance of 75% "
    "in each registered course during the semester to be eligible for End Semester Exams. "
    "Students having attendance between 65% and 74% due to medical reasons may apply for condonation. "
    "Attendance below 65% strictly results in an NE grade."
)

for size, overlap in [(400, 40), (800, 80), (1200, 120)]:
    splitter = RecursiveTextSplitter(chunk_size=size, chunk_overlap=overlap)
    chunks = splitter.split_text(test_text)
    print(f"Chunk Size: {size}, Overlap: {overlap} -> {len(chunks)} chunks produced")

## 2. Ingest Sample Campus Documents

In [ ]:
ingestor = DocumentIngestor(chunk_size=800, chunk_overlap=80)
chunks = ingestor.load_directory("../data/sample_docs")
print(f"Total loaded chunks: {len(chunks)}")

## 3. ChromaDB Persistent Upsert & Semantic Search

In [ ]:
store = ChromaVectorStore(persist_dir="../chroma_db")
store.clear()
store.add_chunks(chunks)
print(f"Chroma collection count: {store.count()}")

retriever = CampusRetriever(store)
query = "Where is the finance office located?"
hits, emb_t, ret_t = retriever.retrieve(query, top_k=2)
print(f"Retrieved {len(hits)} hits in {emb_t:.4f}s + {ret_t:.4f}s:")
for h in hits:
    print(f" - [{h['chunk_id']}] Similarity: {h['similarity']:.3f} | Text: {h['text'][:80]}...")

## 4. Response Generation with Strict Prompting

In [ ]:
generator = CampusGenerator()
context = retriever.format_context(hits)
res = generator.generate(query=query, context=context)
print("Response:", res["text"])
print("Latency:", res["latency"], "s")